# Weather Data Ingestion Pipeline

## Objective

In the previous notebook, we explored the Open-Meteo Historical Weather API and identified the weather attributes required for our project.

This notebook focuses on loading historical weather observations into Snowflake.

## Workflow

1. Create Snowflake Session
2. Configure API Parameters
3. Extract Historical Weather Data
4. Transform JSON Response
5. Add Metadata Columns
6. Create Target Snowflake Table
7. Load Data into Snowflake
8. Validate Data Load

File References:

- Snowflake Connection (`src/utils/snowflake_connection.py`)
- API exploration Notebook (`notebooks/Weather_API_Exploration.ipynb`)

In [1]:
# ==========================================================
# Project Imports
# ==========================================================

import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root / "Snowpark") not in sys.path:
    sys.path.append(str(project_root / "Snowpark"))

In [2]:
# ==========================================================
# Create Snowflake Session
# File: src/utils/snowflake_connection.py
# ==========================================================

from src.utils.snowflake_connection import get_session

session = get_session("dev")

session

try init connection snowflake
{'user': 'LPDG_ASYED', 'account': 'KPCKWFG-LPDG_AWS_FR', 'role': 'SNOWFLAKE_MASTERCLASS', 'warehouse': 'WH_LPDG_RGMCET_MASTERCLASS', 'database': 'DB_LPDG_RGMCET_MASTERCLASS', 'password': 'Syedhameed@786', 'schema': 'SCH_LPDG_RGMCET_MASTERCLASS'}


## Configure Historical Weather Extraction

The weather variables selected in the exploration notebook will now be used to retrieve historical observations.

For demonstration purposes, we will retrieve approximately 60 days of hourly weather data.

In [3]:
# ==========================================================
# Configure API Parameters
# ==========================================================

from datetime import date, timedelta

LATITUDE = 17.3850
LONGITUDE = 78.4867

END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=60)

WEATHER_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "pressure_msl",
    "wind_speed_10m",
    "precipitation",
    "cloud_cover"
]

In [4]:
# ==========================================================
# Build API Request
# ==========================================================

import requests

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

URL = (
    f"{BASE_URL}"
    f"?latitude={LATITUDE}"
    f"&longitude={LONGITUDE}"
    f"&start_date={START_DATE}"
    f"&end_date={END_DATE}"
    f"&hourly={','.join(WEATHER_VARIABLES)}"
)

print(URL)

https://archive-api.open-meteo.com/v1/archive?latitude=17.385&longitude=78.4867&start_date=2026-07-17&end_date=2026-09-15&hourly=temperature_2m,relative_humidity_2m,pressure_msl,wind_speed_10m,precipitation,cloud_cover


In [5]:
# ==========================================================
# Build API Request
# ==========================================================

import requests

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

URL = (
    f"{BASE_URL}"
    f"?latitude={LATITUDE}"
    f"&longitude={LONGITUDE}"
    f"&start_date={START_DATE}"
    f"&end_date={END_DATE}"
    f"&hourly={','.join(WEATHER_VARIABLES)}"
)

print(URL)

https://archive-api.open-meteo.com/v1/archive?latitude=17.385&longitude=78.4867&start_date=2026-07-17&end_date=2026-09-15&hourly=temperature_2m,relative_humidity_2m,pressure_msl,wind_speed_10m,precipitation,cloud_cover


In [7]:
# ==========================================================
# Retrieve Historical Weather Data
# ==========================================================

response = requests.get(URL)

response.raise_for_status()

weather_json = response.json()

print("Weather data retrieved successfully")

Weather data retrieved successfully


## Transform API Response

The API response is returned in JSON format.

For analytics and machine learning workflows, the nested JSON structure will be transformed into a tabular DataFrame.

In [11]:
# ==========================================================
# Convert JSON to DataFrame
# ==========================================================

import pandas as pd

weather_df = pd.DataFrame(weather_json["hourly"])

weather_df.head()

,time,temperature_2m,relative_humidity_2m,pressure_msl,wind_speed_10m,precipitation,cloud_cover
0,2026-07-17T00:00,24.7,78,1007.4,11.0,0.1,100
1,2026-07-17T01:00,24.5,80,1008.1,10.4,0.1,100
2,2026-07-17T02:00,25.0,79,1008.6,11.6,0.2,100
3,2026-07-17T03:00,25.4,79,1009.0,13.1,0.2,100
4,2026-07-17T04:00,25.4,81,1009.3,11.4,0.4,100


In [12]:
# ==========================================================
# Rename Columns
# ==========================================================

weather_df.rename(
    columns={
        "time": "timestamp",
        "temperature_2m": "temperature",
        "relative_humidity_2m": "humidity",
        "pressure_msl": "pressure",
        "wind_speed_10m": "wind_speed"
    },
    inplace=True
)

weather_df.head()

,timestamp,temperature,humidity,pressure,wind_speed,precipitation,cloud_cover
0,2026-07-17T00:00,24.7,78,1007.4,11.0,0.1,100
1,2026-07-17T01:00,24.5,80,1008.1,10.4,0.1,100
2,2026-07-17T02:00,25.0,79,1008.6,11.6,0.2,100
3,2026-07-17T03:00,25.4,79,1009.0,13.1,0.2,100
4,2026-07-17T04:00,25.4,81,1009.3,11.4,0.4,100


## Add Metadata Columns

In addition to weather observations, metadata describing the source location and ingestion process will also be stored.

These fields are useful for:

- Data lineage
- Auditing
- Troubleshooting
- Multi-location expansion in future projects

In [13]:
# ==========================================================
# Add Metadata Columns
# ==========================================================

from datetime import datetime

weather_df["latitude"] = weather_json["latitude"]
weather_df["longitude"] = weather_json["longitude"]
weather_df["timezone"] = weather_json["timezone"]
weather_df["elevation"] = weather_json["elevation"]

weather_df.head()

,timestamp,temperature,humidity,pressure,wind_speed,precipitation,cloud_cover,latitude,longitude,timezone,elevation
0,2026-07-17T00:00,24.7,78,1007.4,11.0,0.1,100,17.398945,78.457085,GMT,505.0
1,2026-07-17T01:00,24.5,80,1008.1,10.4,0.1,100,17.398945,78.457085,GMT,505.0
2,2026-07-17T02:00,25.0,79,1008.6,11.6,0.2,100,17.398945,78.457085,GMT,505.0
3,2026-07-17T03:00,25.4,79,1009.0,13.1,0.2,100,17.398945,78.457085,GMT,505.0
4,2026-07-17T04:00,25.4,81,1009.3,11.4,0.4,100,17.398945,78.457085,GMT,505.0


## Create Target Snowflake Table

The table structure is based on the schema identified during the exploration phase.

Column comments are included to improve discoverability and maintainability.

In [ ]:
# ==========================================================
# Create Target Table
# ==========================================================

create_table_sql = """
CREATE OR REPLACE TABLE TBL_WEATHER_DATA (

    TIMESTAMP TIMESTAMP_NTZ
        COMMENT 'Observation timestamp',

    TEMPERATURE FLOAT
        COMMENT 'Temperature at 2 meters',

    HUMIDITY INTEGER
        COMMENT 'Relative humidity percentage',

    PRESSURE FLOAT
        COMMENT 'Mean sea level pressure',

    WIND_SPEED FLOAT
        COMMENT 'Wind speed at 10 meters',

    PRECIPITATION FLOAT
        COMMENT 'Hourly precipitation',

    CLOUD_COVER INTEGER
        COMMENT 'Cloud cover percentage',

    LATITUDE FLOAT
        COMMENT 'Location latitude',

    LONGITUDE FLOAT
        COMMENT 'Location longitude',

    TIMEZONE STRING
        COMMENT 'Timezone of source location',

    ELEVATION FLOAT
        COMMENT 'Elevation above sea level',

    LOAD_TIMESTAMP TIMESTAMP_NTZ
        COMMENT 'Record ingestion timestamp'
)
"""

In [15]:
session.sql(create_table_sql).collect()

print("Table created successfully")

Table created successfully


## Load Data into Snowflake

The transformed Pandas DataFrame will now be converted into a Snowpark DataFrame and persisted into the target Snowflake table.

In [16]:
# ==========================================================
# Convert Pandas DataFrame to Snowpark DataFrame
# ==========================================================

snowpark_df = session.create_dataframe(weather_df)

snowpark_df.show(5)

TypeError: create_dataframe() function only accepts data as a list, tuple or a pandas DataFrame.